## Konfiguration

In [ ]:
# ============================================================
# Konfiguration (zentral & leicht anpassbar)
# Alle Stellschrauben an einem Ort -> kein Suchen mehr quer durch den Code.
# ============================================================

# Reproduzierbarkeit
SEED = 42

# Pfade
DATA_PATH     = '../Datasets/Iris.csv'
MODEL_PATH    = '../Models/iris_net_Optuna.pth'
ENCODER_PATH  = '../Models/label_encoder_Optuna.pkl'
SCALER_PATH   = '../Models/scaler_Optuna.pkl'
METADATA_PATH = '../Models/metadata_Optuna.json'

# Optuna-Steuerung
N_TRIALS  = 50           # Anzahl der Hyperparameter-Kombinationen, die Optuna testet
DIRECTION = 'maximize'   # Optimierungsziel: Validation Accuracy maximieren

# Feste Trainingsparameter (werden von Optuna NICHT veraendert)
MAX_EPOCHS  = 100
PATIENCE    = 10         # Early-Stopping-Geduld (Epochen ohne Val-Accuracy-Verbesserung)
SCALER_TYPE = 'standard' # Scaler: 'standard' | 'minmax' | 'robust'

# Suchraum-Grenzen (werden in der Objective-Funktion verwendet)
HIDDEN_DIM_MIN  = 4
HIDDEN_DIM_MAX  = 64
LR_MIN          = 1e-4
LR_MAX          = 1e-1
BATCH_SIZE_OPTS = [8, 16, 32, 64]

## Imports

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import LabelEncoder, StandardScaler, MinMaxScaler, RobustScaler
import numpy as np
import pandas as pd
import joblib
import json
import copy
import random
import os
import optuna  # Hyperparameter-Optimierungs-Framework: automatische Suche im Parameterraum

# Optuna-interne Fortschrittsausgaben auf Warnungen reduzieren -> uebersichtlichere Ausgabe
optuna.logging.set_verbosity(optuna.logging.WARNING)

## Reproduzierbarkeit

In [ ]:
# --- Reproduzierbarkeit: alle relevanten Zufallsquellen fixieren ---
# Ohne torch.manual_seed waeren Gewichts-Initialisierung und das Shuffeln im
# DataLoader bei jedem Lauf anders -> nicht reproduzierbare Ergebnisse.
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

## Daten laden & aufteilen

In [ ]:
# 1. Laden des Datensatzes
df = pd.read_csv(DATA_PATH)

# 2. Features (X) und Target (y) trennen
X = df.drop('species', axis=1).values
y = df['species'].values

# Feature-Namen direkt aus den Daten ableiten (verbindliche Reihenfolge fuer die Inferenz!)
feature_names = df.drop('species', axis=1).columns.tolist()

# LabelEncoder sortiert die gefundenen Text-Kategorien standardmaessig alphabetisch
le = LabelEncoder()
y = le.fit_transform(y)

# 3. Aufteilen des Datensatzes (Train / Val / Test) - beide Splits mit demselben SEED
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=SEED)
X_train, X_val, y_train, y_val   = train_test_split(X_train, y_train, test_size=0.2, random_state=SEED)

# --- Feature Scaling ---
# Fuer neuronale Netze ist Skalierung essenziell fuer die Konvergenz. Gaengige Methoden:
# 1. StandardScaler (Standardisierung): Mittelwert = 0, Standardabweichung = 1 (Standard fuer NNs)
# 2. MinMaxScaler (Normalisierung): Skaliert Werte starr in einen Bereich (meist 0 bis 1)
# 3. RobustScaler: Nutzt Median und Quartile, sehr robust gegenueber Ausreissern (Outliers)
# Die Auswahl erfolgt zentral ueber SCALER_TYPE in der Konfiguration:
if SCALER_TYPE == 'standard':
    scaler = StandardScaler()
elif SCALER_TYPE == 'minmax':
    scaler = MinMaxScaler()
elif SCALER_TYPE == 'robust':
    scaler = RobustScaler()
else:
    raise ValueError(f"Unbekannter SCALER_TYPE: {SCALER_TYPE!r} (erlaubt: 'standard', 'minmax', 'robust')")

# WICHTIG (Vermeidung von Data Leakage): 'fit_transform' NUR auf Trainingsdaten anwenden!
# Der Scaler lernt hier die Parameter (z.B. Mittelwert/Varianz) der Trainingsdaten.
X_train = scaler.fit_transform(X_train)

# Auf Validierungs- und Testdaten NUR 'transform' anwenden!
# Sie werden mit den aus den Trainingsdaten gelernten Parametern skaliert.
X_val  = scaler.transform(X_val)
X_test = scaler.transform(X_test)
# -----------------------------------------

# 4. Umwandeln der skalierten Daten in PyTorch-Tensoren
X_train = torch.from_numpy(X_train).float()
X_test  = torch.from_numpy(X_test).float()
y_train = torch.from_numpy(y_train).long()
y_test  = torch.from_numpy(y_test).long()
X_val   = torch.from_numpy(X_val).float()
y_val   = torch.from_numpy(y_val).long()

# Dimensionen aus den Daten ableiten (statt hartkodiert -> keine Inkonsistenzen mehr)
input_dim  = X_train.shape[1]  # Anzahl Eingabe-Merkmale
output_dim = len(le.classes_)  # Anzahl Klassen

# TensorDataset buendelt Features und Labels zu einem iterierbaren Datensatz
train_dataset = TensorDataset(X_train, y_train)

## Optuna Objective-Funktion

In [ ]:
def objective(trial):
    """Optuna-Objective: wird fuer jeden Trial einmal aufgerufen.

    Ein 'Trial' ist ein einzelner Optimierungsversuch mit einer konkreten
    Hyperparameter-Kombination, die Optuna automatisch vorschlaegt.
    Die Funktion gibt die Validation Accuracy zurueck, die Optuna maximieren soll.
    """

    # --- Hyperparameter-Vorschlaege durch Optuna ---
    # 'suggest_int' waehlt eine ganzzahlige Zahl aus dem angegebenen Bereich
    hidden_dim    = trial.suggest_int('hidden_dim', HIDDEN_DIM_MIN, HIDDEN_DIM_MAX)
    # 'suggest_float' mit log=True sucht auf einer logarithmischen Skala (sinnvoll fuer Lernraten)
    learning_rate = trial.suggest_float('learning_rate', LR_MIN, LR_MAX, log=True)
    # 'suggest_categorical' waehlt einen Wert aus einer vorgegebenen Liste
    batch_size    = trial.suggest_categorical('batch_size', BATCH_SIZE_OPTS)

    # 1. Netz mit trial-spezifischer Architektur aufbauen (identisch zur Vorlage)
    net = nn.Sequential(
        nn.Linear(input_dim, hidden_dim),   # Eingabeschicht -> versteckte Schicht
        nn.ReLU(),                          # Aktivierungsfunktion: ReLU
        nn.Linear(hidden_dim, output_dim)   # versteckte Schicht -> Ausgabeschicht
    )

    # 2. DataLoader mit trial-spezifischer batch_size erstellen
    # Generator mit SEED -> reproduzierbares Shuffeln ueber alle Trials hinweg
    g = torch.Generator()
    g.manual_seed(SEED)
    train_loader = DataLoader(
        train_dataset, batch_size=batch_size, shuffle=True, generator=g
    )

    # 3. Optimizer und Verlustfunktion definieren
    optimizer = optim.AdamW(net.parameters(), lr=learning_rate)  # AdamW mit trial-spezifischer Lernrate
    criterion = nn.CrossEntropyLoss()                            # Standardverlust fuer Klassifikation

    # --- Setup fuer Early Stopping innerhalb des Trials ---
    # Early Stopping ueberwacht hier die Validation ACCURACY (nicht den Loss wie in der Vorlage),
    # weil Optuna die Accuracy maximieren soll -> konsistentes Optimierungsziel.
    patience_counter    = 0      # Zaehler fuer Epochen ohne Verbesserung
    best_val_accuracy   = 0.0   # beste bisher gesehene Validation Accuracy dieses Trials
    best_model_weights  = None  # Sicherung der besten Gewichte

    # 5. Trainingsschleife ueber MAX_EPOCHS Epochen
    for epoch in range(MAX_EPOCHS):
        net.train()  # Trainingsmodus aktivieren (relevant fuer Dropout/BatchNorm)
        kumulierter_train_loss = 0.0  # Variable zum Aufsummieren des Batch-Losses

        for batch_data in train_loader:  # Schleife ueber die Batches aus dem DataLoader
            batch_X = batch_data[0]
            batch_y = batch_data[1]
            optimizer.zero_grad()
            outputs = net(batch_X)
            loss = criterion(outputs, batch_y)
            loss.backward()
            optimizer.step()
            kumulierter_train_loss += loss.item()  # Loss aufsummieren

        # Validierung nach jeder Epoche
        net.eval()
        with torch.no_grad():
            val_out = net(X_val)
            _, val_predicted = torch.max(val_out, 1)  # Klasse mit hoechstem Logit
            # Validation Accuracy als Anteil korrekt klassifizierter Beispiele
            val_accuracy = accuracy_score(y_val.numpy(), val_predicted.numpy())

        # --- Early Stopping Logik (ueberwacht Validation Accuracy) ---
        # Ist die aktuelle Validation Accuracy die beste, die wir je gesehen haben?
        if val_accuracy > best_val_accuracy:
            best_val_accuracy  = val_accuracy
            patience_counter   = 0  # Zaehler zuruecksetzen
            # Tiefe Kopie der Gewichte sichern, damit wir spaeter den besten Zustand laden koennen
            best_model_weights = copy.deepcopy(net.state_dict())
        else:
            patience_counter += 1  # Keine Verbesserung -> Zaehler erhoehen

        # Zu lange keine Verbesserung -> Trial vorzeitig beenden
        if patience_counter >= PATIENCE:
            break
        # ----------------------------------------------

    # 6. Beste Validation Accuracy dieses Trials zurueckgeben
    # Optuna vergleicht diesen Wert ueber alle Trials und merkt sich das Maximum.
    return best_val_accuracy

## Optuna Study ausführen

In [ ]:
# Study erstellen: TPESampler fuer effiziente Suche, MedianPruner nicht aktiv
# (Pruning ist auf Trial-Ebene nicht konfiguriert, da Early Stopping
#  bereits schlechte Trials intern abbricht)
# TPESampler (Tree-structured Parzen Estimator): lernt aus bisherigen Trials,
# welche Hyperparameter-Bereiche vielversprechend sind -> effizienter als Random Search.
sampler = optuna.samplers.TPESampler(seed=SEED)

# Eine 'Study' ist die uebergeordnete Optimierungskampagne, die alle Trials verwaltet
study = optuna.create_study(direction=DIRECTION, sampler=sampler)

# N_TRIALS Trials starten: Optuna ruft objective() N_TRIALS-mal mit verschiedenen Params auf
study.optimize(objective, n_trials=N_TRIALS)

# Ergebnisse des besten Trials ausgeben
print(f"Beste Trial-Nummer : {study.best_trial.number}")
print(f"Beste Val-Accuracy : {study.best_value:.4f}")
print("Beste Hyperparameter:")
print(f"  hidden_dim    : {study.best_params['hidden_dim']}")
print(f"  learning_rate : {study.best_params['learning_rate']:.2e}")
print(f"  batch_size    : {study.best_params['batch_size']}")

## Bestes Modell neu trainieren

In [ ]:
# Beste Hyperparameter aus der abgeschlossenen Study auslesen
best_hidden_dim    = study.best_params['hidden_dim']
best_learning_rate = study.best_params['learning_rate']
best_batch_size    = study.best_params['batch_size']

# Finales Modell mit den besten Hyperparametern neu aufbauen
# (kein Recycling der Trial-Gewichte -> sauberer, reproduzierbarer Neustart)
net = nn.Sequential(
    nn.Linear(input_dim, best_hidden_dim),   # Eingabeschicht -> versteckte Schicht
    nn.ReLU(),                               # Aktivierungsfunktion: ReLU
    nn.Linear(best_hidden_dim, output_dim)   # versteckte Schicht -> Ausgabeschicht
)

# DataLoader mit dem besten batch_size erstellen
g = torch.Generator()
g.manual_seed(SEED)  # Generator mit Seed -> reproduzierbares Shuffeln
train_loader = DataLoader(
    train_dataset, batch_size=best_batch_size, shuffle=True, generator=g
)

# Optimizer und Verlustfunktion mit den besten Parametern definieren
optimizer = optim.AdamW(net.parameters(), lr=best_learning_rate)
criterion = nn.CrossEntropyLoss()  # Standardverlust fuer Klassifikation

# --- Setup fuer Early Stopping des finalen Trainings ---
patience_counter   = 0      # Zaehler fuer die Epochen ohne Verbesserung
best_val_accuracy  = 0.0   # beste bisher gesehene Validation Accuracy
best_model_weights = None  # hier speichern wir die besten Gewichte

# Listen zum Speichern der Historie (ideal fuer spaetere Plots, z.B. mit matplotlib)
history = {'train_loss': [], 'val_loss': []}

# Finales Training mit denselben Schleifen- und Early-Stopping-Mechanismen wie in der Objective
for epoch in range(MAX_EPOCHS):
    net.train()  # Trainingsmodus aktivieren (relevant fuer Dropout/BatchNorm)
    kumulierter_train_loss = 0.0  # Variable zum Aufsummieren des Batch-Losses

    for batch_data in train_loader:  # Schleife ueber die Batches aus dem DataLoader
        batch_X = batch_data[0]
        batch_y = batch_data[1]
        optimizer.zero_grad()
        outputs = net(batch_X)
        loss = criterion(outputs, batch_y)
        loss.backward()
        optimizer.step()
        kumulierter_train_loss += loss.item()  # Loss aufsummieren

    durchschnittlicher_train_loss = kumulierter_train_loss / len(train_loader)
    history['train_loss'].append(durchschnittlicher_train_loss)

    # Validierung nach jeder Epoche
    net.eval()
    with torch.no_grad():
        val_out  = net(X_val)
        val_loss = criterion(val_out, y_val).item()  # direkt als float speichern
        _, val_predicted = torch.max(val_out, 1)     # Klasse mit hoechstem Logit
        val_accuracy = accuracy_score(y_val.numpy(), val_predicted.numpy())

    history['val_loss'].append(val_loss)
    print(f'Epoch {epoch:3d} | Loss: {durchschnittlicher_train_loss:.4f} | Val Loss: {val_loss:.4f}')

    # --- Early Stopping Logik ---
    # Ist die aktuelle Validation Accuracy die beste, die wir je gesehen haben?
    if val_accuracy > best_val_accuracy:
        best_val_accuracy  = val_accuracy
        patience_counter   = 0  # Zaehler zuruecksetzen
        best_model_weights = copy.deepcopy(net.state_dict())
    else:
        patience_counter += 1  # Keine Verbesserung -> Zaehler erhoehen

    # Zu lange keine Verbesserung -> Abbruch!
    if patience_counter >= PATIENCE:
        print(f"\n--- Early Stopping ausgeloest in Epoche {epoch + 1} ---")
        print(f"Beste Validation Accuracy war: {best_val_accuracy:.4f}")
        break
    # ----------------------------------------------

# Wir laden die besten Gewichte zurueck in das Modell.
# Ohne diesen Schritt haetten wir das Modell im Zustand des Overfittings!
net.load_state_dict(best_model_weights)
print("\nBeste Modellgewichte wurden wiederhergestellt.")

## Evaluation auf Testdaten

In [ ]:
# Auswerten des neuronalen Netzes auf den Testdaten
net.eval()
with torch.no_grad():
    outputs = net(X_test)
    _, predicted = torch.max(outputs, 1)
    accuracy = accuracy_score(y_test, predicted)
    print('Testgenauigkeit: ', accuracy)

## Artefakte speichern

In [ ]:
# Sicherstellen, dass das Zielverzeichnis existiert
os.makedirs('Models', exist_ok=True)

# --- Abspeichern: Modell, Encoder, Scaler + Metadaten ---
torch.save(net.state_dict(), MODEL_PATH)
joblib.dump(le, ENCODER_PATH)
joblib.dump(scaler, SCALER_PATH)

# Metadaten beschreiben, WIE das Modell zu benutzen ist: Feature-Reihenfolge,
# Klassen-Mapping, Skalierungstyp, Architektur und Version. Ohne sie muss
# beim spaeteren Laden alles erraten werden.
metadata = {
    'feature_names': feature_names,
    'class_names':   le.classes_.tolist(),
    'class_mapping': {str(i): name for i, name in enumerate(le.classes_)},
    'scaler_type':   SCALER_TYPE,
    'architecture': {
        'input_dim':  int(input_dim),
        'hidden_dim': int(best_hidden_dim),   # bester Wert aus Optuna
        'output_dim': int(output_dim),
    },
    'best_val_accuracy': float(best_val_accuracy),  # beste Validation Accuracy des finalen Trainings
    'test_accuracy':     float(accuracy),
    'seed':              SEED,
    'torch_version':     torch.__version__,
    # Optuna-Block: einzige strukturelle Erweiterung gegenueber der Vorlage
    'optuna': {
        'n_trials':   N_TRIALS,
        'best_trial': study.best_trial.number,
        'best_params': {
            'hidden_dim':    int(best_hidden_dim),
            'learning_rate': float(best_learning_rate),
            'batch_size':    int(best_batch_size),
        },
    },
}
with open(METADATA_PATH, 'w', encoding='utf-8') as f:
    json.dump(metadata, f, indent=2, ensure_ascii=False)
print('Metadaten gespeichert:', METADATA_PATH)

## Artefakte laden & prüfen

In [ ]:
# Wiederladen von Modell + Artefakten (aus den Metadaten rekonstruiert)
import torch
import torch.nn as nn
import joblib
import json

# Metadaten ZUERST laden -> daraus die Architektur rekonstruieren
with open(METADATA_PATH, 'r', encoding='utf-8') as f:
    metadata = json.load(f)
arch = metadata['architecture']

# WICHTIG: Erst die Architektur definieren, DANN die Gewichte laden.
# (Wuerden wir die Gewichte zuerst laden und danach das Netz neu erstellen,
#  wuerden die geladenen Gewichte sofort ueberschrieben.)
net = nn.Sequential(
    nn.Linear(arch['input_dim'], arch['hidden_dim']),  # Eingabeschicht -> versteckte Schicht
    nn.ReLU(),                                         # Aktivierungsfunktion: ReLU
    nn.Linear(arch['hidden_dim'], arch['output_dim'])  # versteckte Schicht -> Ausgabeschicht
)
net.load_state_dict(torch.load(MODEL_PATH, map_location=torch.device('cpu')))
net.eval()  # Inferenzmodus: Dropout/BatchNorm werden deaktiviert

le     = joblib.load(ENCODER_PATH)  # LabelEncoder fuer Klassen-Rueckuebersetzung
scaler = joblib.load(SCALER_PATH)   # Scaler fuer identische Vorverarbeitung wie im Training

# Alle lernbaren Parameter des Netzes ausgeben (Gewichte & Biases)
for k, v in net.named_parameters():
    print(k, v)

## Inferenz: Klassen-Vorhersage

In [ ]:
# Vorhersage mit dem trainierten Netz
# Statt interaktivem input() definieren wir die Eingaben als Array. Das ist
# reproduzierbar, skript-/batch-tauglich und leicht erweiterbar: einfach
# weitere Zeilen ergaenzen. Die Spalten-Reihenfolge MUSS metadata['feature_names'] entsprechen.
import numpy as np

print('Erwartete Merkmal-Reihenfolge:', metadata['feature_names'])

# Beispiel-Eingaben (eine Zeile = ein Datenpunkt)
beispiel_eingaben = np.array([
    [5.1, 3.5, 1.4, 0.2],
    [6.7, 3.0, 5.2, 2.3],
])

# Skalierung mit demselben (geladenen) Scaler wie im Training
eingabe_skaliert = scaler.transform(beispiel_eingaben)

# Erstellen eines Tensors aus den Eingabewerten
inputs = torch.tensor(eingabe_skaliert, dtype=torch.float32)

# Vorhersage treffen
with torch.no_grad():
    outputs = net(inputs)
    _, predicted = torch.max(outputs, 1)  # Index der Klasse mit dem hoechsten Logit

# Ausgabe der Klassifizierungsaussage je Datenpunkt
for i, idx in enumerate(predicted):
    klasse = le.inverse_transform([idx.item()])[0]  # numerischen Index -> Klassenname
    print(f"Eingabe {beispiel_eingaben[i].tolist()} -> Klasse: {klasse}")

## Inferenz: Wahrscheinlichkeiten

In [ ]:
# Vorhersagen als Wahrscheinlichkeiten (nutzt 'inputs' aus der vorherigen Zelle)
import torch.nn.functional as F

# Vorhersage des Netzes
with torch.no_grad():
    output = net(inputs)
    _, max_index = torch.max(output, 1)  # Index der wahrscheinlichsten Klasse

# Klassenwahrscheinlichkeiten berechnen
# Softmax wandelt rohe Logits in Wahrscheinlichkeiten um (Summe = 1.0)
probabilities = F.softmax(output, dim=1)

for i in range(len(inputs)):
    klasse = le.inverse_transform([max_index[i].item()])[0]  # numerischen Index -> Klassenname
    wahrscheinlichkeiten = {
        le.classes_[c]: round(float(probabilities[i, c]), 4)
        for c in range(len(le.classes_))
    }
    print(f"Datenpunkt {i}: Klasse = {klasse} | Wahrscheinlichkeiten = {wahrscheinlichkeiten}")